# exp02 + exp03: 개선된 후처리 & Self-Consistency

**가설 H1**: 토큰 한도 2048 + 소수점-안전 추출로 잘림/오추출 문제 해결 (+2~4%p)
**가설 H2**: 같은 문제를 8번 풀게 해 다수결(Self-Consistency)로 답 선정 (+5~10%p)

검증은 train에서 뗀 **고정 500문제** (random_state=123, 이후 학습에서 제외)로 한다.

## 실행 옵션 (아래 셀의 플래그로 켜고 끄기)
- `RUN_GREEDY_EVAL` (exp02, 약 2시간): 개선판 greedy를 검증 500문제로 평가
- `RUN_SC_EVAL` (exp03, 약 4~5시간): Self-Consistency n=8을 같은 500문제로 평가
- `RUN_SUBMISSION` (약 8시간, **별도 세션 권장**): 리더보드 1,000문제를 SC로 풀어 submission.csv 생성

Kaggle 설정: **GPU T4 x2**, **Internet On**

In [ ]:
# ── 0. 실행 플래그 ──
RUN_GREEDY_EVAL = True    # exp02
RUN_SC_EVAL = True        # exp03
RUN_SUBMISSION = False    # 평가 결과 확인 후 별도 세션에서 True로

SC_N = 8                  # Self-Consistency 샘플 수
SC_TEMP = 0.7             # 샘플링 온도
MAX_NEW_TOKENS = 2048     # exp01의 1024에서 증가 (H1)
VAL_N = 500               # 고정 검증 세트 크기
VAL_SEED = 123            # 절대 바꾸지 말 것 (실험 간 비교 기준)

In [ ]:
# ── 1. 재현용 환경 기록 (검증 제출물 대비) ──
import subprocess
freeze = subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout
with open('/kaggle/working/requirements_freeze.txt', 'w') as f:
    f.write(freeze)
print('환경 기록 완료: requirements_freeze.txt (Output에서 다운로드해 repo에 보관)')

In [ ]:
# ── 2. 데이터: 고정 검증 세트 분리 ──
import glob
import pandas as pd

def find_csv(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} 없음 — Input 패널에서 대회 데이터 확인'
    return hits[0]

train_df = pd.read_csv(find_csv('deep_chal_math_train.csv'))
lb_df = pd.read_csv(find_csv('deep_chal_math_leaderboard.csv'))
train_df.columns = train_df.columns.str.strip()
lb_df.columns = lb_df.columns.str.strip()

val_df = train_df.sample(VAL_N, random_state=VAL_SEED).reset_index(drop=True)
# 검증 세트 id 목록 저장 → 나중에 SFT 학습 데이터에서 제외할 때 사용
val_df[['id']].to_csv('/kaggle/working/val_ids.csv', index=False)
print(f'검증 세트 {len(val_df)}문제 고정 (val_ids.csv 저장)')

In [ ]:
# ── 3. 모델 로드 ──
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto')
model.eval()
print('모델 로드 완료')

In [ ]:
# ── 4. 개선된 답 추출 (H1) ──
# exp01 문제: '38.585'에서 585를 뽑음. 소수를 통째로 매칭해 정수가 아니면 버린다.
import re

def parse_int(s):
    s = re.sub(r'[,\s$]', '', s)
    s = re.sub(r'\\[!,;:]', '', s)
    s = s.strip('.')
    try:
        return int(s)
    except ValueError:
        pass
    try:
        f = float(s)
        if abs(f - round(f)) < 1e-6:
            return int(round(f))
    except (ValueError, OverflowError):
        pass
    return None

# 소수·분수까지 통째로 잡는 패턴 (소수의 꼬리만 떼어 정수로 착각하는 것 방지)
NUM_PAT = re.compile(r'-?\d[\d,]*(?:\.\d+)?')

def extract_answer(text):
    # 1순위: \boxed{...} (중첩 중괄호 1단계까지 허용)
    boxed = re.findall(r'\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}', text)
    for cand in reversed(boxed):
        # \frac{a}{b} 형태면 나눗셈 시도
        frac = re.match(r'^\\d?frac\{(-?\d+)\}\{(-?\d+)\}$', cand.strip())
        if frac:
            a, b = int(frac.group(1)), int(frac.group(2))
            if b != 0 and a % b == 0:
                return a // b
        val = parse_int(cand)
        if val is not None:
            return val
        # boxed 안에 다른 문자가 섞여 있으면 그 안의 숫자만
        inner = NUM_PAT.findall(cand)
        for c in reversed(inner):
            val = parse_int(c)
            if val is not None:
                return val
    # 2순위: 본문의 마지막 '정수' (소수는 통째로 매칭돼 걸러짐)
    for cand in reversed(NUM_PAT.findall(text)):
        val = parse_int(cand)
        if val is not None:
            return val
    return None  # SC 다수결에서 무효표 처리하기 위해 0 대신 None

assert extract_answer('answer is \\boxed{-42}.') == -42
assert extract_answer('38.585 \\ldots approx') is None or extract_answer('38.585 approx x') != 585
assert extract_answer('total 1,234 dollars') == 1234
assert extract_answer('\\boxed{\\frac{10}{2}}') == 5
print('답 추출 함수 OK')

In [ ]:
# ── 5. 생성 함수 (greedy / 샘플링 겸용) ──
from tqdm.auto import tqdm

SYSTEM_PROMPT = (
    'You are an expert competition mathematician. '
    'Solve the problem step by step. '
    'The final answer is always an integer. '
    'Put your final integer answer inside \\boxed{}.'
)

def build_prompt(question):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(questions, n=1, temperature=None, batch_problems=4):
    """문제 리스트 → 문제별 풀이 n개의 리스트. n=1이면 greedy."""
    results = []
    for i in tqdm(range(0, len(questions), batch_problems)):
        batch = [build_prompt(q) for q in questions[i:i + batch_problems]]
        enc = tokenizer(batch, return_tensors='pt', padding=True).to(model.device)
        kwargs = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id)
        if n == 1:
            kwargs['do_sample'] = False
        else:
            kwargs.update(do_sample=True, temperature=temperature, top_p=0.8,
                          num_return_sequences=n)
        gen = model.generate(**enc, **kwargs)
        new = gen[:, enc['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(new, skip_special_tokens=True)
        # num_return_sequences로 늘어난 출력을 문제별로 묶는다
        for j in range(len(batch)):
            results.append(texts[j * n:(j + 1) * n])
    return results

from collections import Counter

def majority_vote(answers):
    """None(무효)을 뺀 다수결. 동률이면 먼저 나온 것."""
    votes = [a for a in answers if a is not None]
    if not votes:
        return 0
    return Counter(votes).most_common(1)[0][0]

In [ ]:
# ── 6. exp02: greedy 개선판 평가 (검증 500) ──
if RUN_GREEDY_EVAL:
    outs = generate(val_df['question'].tolist(), n=1, batch_problems=16)
    preds = [extract_answer(o[0]) for o in outs]
    acc = sum(1 for p, a in zip(preds, val_df['answer']) if p is not None and int(p) == int(a)) / len(val_df)
    none_rate = sum(1 for p in preds if p is None) / len(preds)
    print(f'[exp02] greedy 개선판 정확도: {acc:.1%} (추출 실패율 {none_rate:.1%})')
    pd.DataFrame({'id': val_df['id'], 'pred': preds, 'answer': val_df['answer'],
                  'output': [o[0] for o in outs]}).to_csv('/kaggle/working/exp02_val_outputs.csv', index=False)
    print('상세 출력 저장: exp02_val_outputs.csv (오답 분석용 — 다운로드 권장)')

In [ ]:
# ── 7. exp03: Self-Consistency n=8 평가 (검증 500) ──
if RUN_SC_EVAL:
    outs = generate(val_df['question'].tolist(), n=SC_N, temperature=SC_TEMP, batch_problems=4)
    preds = [majority_vote([extract_answer(t) for t in o]) for o in outs]
    acc = sum(int(p) == int(a) for p, a in zip(preds, val_df['answer'])) / len(val_df)
    print(f'[exp03] Self-Consistency n={SC_N} 정확도: {acc:.1%}')
    # 몇 표 차이로 이겼는지 분포 (다수결이 실제로 일하는지 확인)
    from collections import Counter as C
    margins = []
    for o in outs:
        votes = [a for a in (extract_answer(t) for t in o) if a is not None]
        if votes:
            margins.append(C(votes).most_common(1)[0][1])
    print('최다 득표수 분포:', dict(sorted(C(margins).items())))

In [ ]:
# ── 8. 리더보드 제출용 (별도 세션에서 RUN_SUBMISSION=True로) ──
if RUN_SUBMISSION:
    outs = generate(lb_df['question'].tolist(), n=SC_N, temperature=SC_TEMP, batch_problems=4)
    preds = [majority_vote([extract_answer(t) for t in o]) for o in outs]
    submission = pd.DataFrame({'id': lb_df['id'], 'answer': preds})
    submission['answer'] = submission['answer'].astype('int64')
    submission.to_csv('/kaggle/working/submission.csv', index=False)
    print('submission.csv 저장 완료')